In [84]:
import pandas as pd
import yaml

In [85]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [86]:
config

{'data': {'raw': {'file1': '../data/raw/df_final_demo.txt',
   'file2': '../data/raw/df_final_experiment_clients.txt',
   'file3': '../data/raw/df_final_web_data_pt_1.txt',
   'file4': '../data/raw/df_final_web_data_pt_2.txt',
   'file5': '../data/raw/final_web_data_merged.csv'},
  'clean': {'file1': '../data/clean/df_demo_cleaned.csv',
   'file2': '../data/clean/df_experiment_clients_cleaned.csv',
   'file3': '../data/clean/df_test_cids.csv',
   'file4': '../data/clean/df_control_cids.csv',
   'file5': '../data/clean/final_web_data_merged_confirmed.csv',
   'file6': '../data/clean/final_web_data_merged_cleaned.csv'}}}

In [87]:
df_w = pd.read_csv(config['data']['clean']['file6'], quotechar='"')
df_test= pd.read_csv(config['data']['clean']['file3'], quotechar='"')
df_cont= pd.read_csv(config['data']['clean']['file4'], quotechar='"')

In [88]:
#find most freq client to get a sense of the data
# top_client = df_w.client_id.value_counts().index[0]
# df_w[(df_w.client_id==top_client) & (df_w.process_step == 'confirm')]
# finished_ids = df_w[df_w.process_step == 'confirm'].client_id

In [89]:
df_wtest = df_w[df_w.client_id.isin(df_test.client_id)]
df_wcontrol = df_w[df_w.client_id.isin(df_cont.client_id)]

In [90]:
def contains_step_error(lst, sub):
    """
    function takes as input a list and looks for a sublist inside
    input: list and sub list
    output boolean
    """
    n = len(sub)
    return any(lst[i:i+n] == sub for i in range(len(lst)-n+1))

In [91]:
patterns = {
    's_1': ['start', 'step_1', 'start'],
    '1_2': ['step_1', 'step_2', 'step_1'],
    '2_3': ['step_2', 'step_3', 'step_2'],
    '3_c': ['step_3', 'confirm', 'step_3']
}
results = {key: set() for key in patterns}

for client_id, group in df_w.groupby('client_id'):
    step_list = group['process_step'].tolist()
    for key, pattern in patterns.items():
        if contains_step_error(step_list, pattern):
            results[key].add(client_id)

total = df_w['client_id'].nunique()
error_rates = {key: len(ids) / total for key, ids in results.items()}
print(error_rates)

{'s_1': 0.14988138344362592, '1_2': 0.054963166437757525, '2_3': 0.055720647604777955, '3_c': 0.0001581554084987722}


In [192]:
steps = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

def get_completion_rates(df):
    total = df['client_id'].nunique()
    rates = {}
    for step in steps:
        clients_reached = df[df['process_step'] == step]['client_id'].nunique()
        rates[step] = clients_reached #/ total
    return rates

In [212]:
def get_error_rate(df):
    patterns = {
        's_1': ['start', 'step_1', 'start'],
        '1_2': ['step_1', 'step_2', 'step_1'],
        '2_3': ['step_2', 'step_3', 'step_2'],
        '3_c': ['step_3', 'confirm', 'step_3']
    }
    results = {key: set() for key in patterns}
    err_ids = set()
    
    for client_id, group in df.groupby('client_id'):
        step_list = group['process_step'].tolist()
        for key, pattern in patterns.items():
            if contains_step_error(step_list, pattern):
                results[key].add(client_id)
                err_ids.add(client_id)
    
    # total = df['client_id'].nunique()
    # err_pc = len(err_ids)/total*100
    # error_rates = {key: len(ids) / total for key, ids in results.items()}
    error_rates = {key: len(ids) for key, ids in results.items()}
    error_rates['err_counts']=len(err_ids)   #err_pc
    results['err_counts']=err_ids
    return error_rates, results

In [213]:
finished_cont = get_completion_rates(df_wcontrol)
finished_test = get_completion_rates(df_wtest)

In [215]:
error_cont, res_cont = get_error_rate(df_wcontrol)
error_test, res_test = get_error_rate(df_wtest)
df_err_c = pd.DataFrame([error_cont])
df_err_t = pd.DataFrame([error_test])
df_err_c, df_err_t

(    s_1  1_2   2_3  3_c  err_counts
 0  2483  895  1689    2        4696,
     s_1   1_2   2_3  3_c  err_counts
 0  4685  1490  1549    5        6957)

In [217]:
from statsmodels.stats.proportion import proportions_ztest
ind=0
for col in res_test.keys():
    count_cont = list(finished_cont.values())[ind]
    count_test = list(finished_test.values())[ind]
    ind+=1
    stat, pval = proportions_ztest([len(res_cont[col]),len(res_test[col])],[count_cont,count_test])
    print(col, stat,pval)

s_1 -22.15193574824145 9.99317749740091e-109
1_2 -7.907584632064118 2.6243072216520052e-15
2_3 7.824377387743825 5.101754893436567e-15
3_c -0.8987194785975681 0.3688020990511328
err_counts -13.195438554019661 9.321083535737845e-40


In [165]:
stat, pval = proportions_ztest([count_cont,count_test], [df_err_c['s_1'][0], df_err_t['s_1'][0]])
pval

np.float64(nan)

In [167]:
[count_cont,count_test], [df_err_c[col][0], df_err_t[col][0]]

([126884, 157687],
 [np.float64(19.9566529259275), np.float64(25.801068090787716)])

In [171]:
df_err_c, df_err_t

(       s_1       1_2       2_3       3_c     err_pc
 0  0.10552  0.038035  0.071778  0.000085  19.956653,
        s_1       1_2       2_3       3_c     err_pc
 0  0.17375  0.055259  0.057447  0.000185  25.801068)

In [227]:
for client_id, group in df_w.groupby('client_id'):
    if group['visit_id'].nunique() > 2:
        print(group.visit_id)

259125    451173196_11661340552_563345
259126    451173196_11661340552_563345
259127    451173196_11661340552_563345
259128    451173196_11661340552_563345
460977     72221519_52513154978_430287
460978     72221519_52513154978_430287
597171    905546080_75813398358_250101
597172    905546080_75813398358_250101
Name: visit_id, dtype: str
206743     379946188_1773022140_107963
394689    633860590_96880450633_976109
394690    633860590_96880450633_976109
394691    633860590_96880450633_976109
484332    753205700_16851596206_134483
484333    753205700_16851596206_134483
484334    753205700_16851596206_134483
484335    753205700_16851596206_134483
484336    753205700_16851596206_134483
Name: visit_id, dtype: str
105389    243444359_78696078676_118990
469366    733093772_49185493415_662403
537580     825109778_55379502512_27372
Name: visit_id, dtype: str
129277    275887696_51740057136_798210
129278    275887696_51740057136_798210
129279    275887696_51740057136_798210
129280    275887696_51

In [235]:
print(df_w[(df_w['visit_id']=='451173196_11661340552_563345')] )
print(df_w[df_w['visit_id']=='72221519_52513154978_430287'])

        client_id             visitor_id                      visit_id  \
259125        805  831412807_82548325803  451173196_11661340552_563345   
259126        805  831412807_82548325803  451173196_11661340552_563345   
259127        805  831412807_82548325803  451173196_11661340552_563345   
259128        805  831412807_82548325803  451173196_11661340552_563345   

       process_step            date_time  
259125        start  2017-06-15 19:11:28  
259126       step_2  2017-06-15 19:09:26  
259127       step_1  2017-06-15 19:09:19  
259128        start  2017-06-15 19:09:13  
        client_id             visitor_id                     visit_id  \
460977        805  831412807_82548325803  72221519_52513154978_430287   
460978        805  831412807_82548325803  72221519_52513154978_430287   

       process_step            date_time  
460977       step_1  2017-06-17 19:23:20  
460978        start  2017-06-17 19:23:17  


In [231]:
df_w.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time'], dtype='str')

In [224]:
df_w.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17
1,9056452,306992881_89423906595,1000165_4190026492_760066,confirm,2017-06-04 01:09:50
2,9056452,306992881_89423906595,1000165_4190026492_760066,step_3,2017-06-04 01:09:13
3,9056452,306992881_89423906595,1000165_4190026492_760066,step_2,2017-06-04 01:07:56
4,9056452,306992881_89423906595,1000165_4190026492_760066,step_1,2017-06-04 01:07:32
